# MAMP-ml on Google Colab

Predict the immunogenicity of receptor–ligand pairs in plants using the pretrained ESM-2-based MAMP-ml model.

**This notebook is four steps:**

1. **Install** `mamp-ml` **and a folding backend** (ColabFold or ESMFold) — up front, before anything else.
2. **Upload** a spreadsheet of receptor–ligand pairs.
3. **Run** `mamp-ml predict` — it folds the receptors automatically *and* runs inference, in one command.
4. **Download** `predictions.csv` with per-row class probabilities.

> **Tip:** Pick *Runtime → Change runtime type → GPU* before running anything. Folding needs a GPU to be fast, and ESM-2 inference is much faster on one too.

## 1. Install MAMP-ml

Installs from the `version2` branch on GitHub. This pulls all Python dependencies (torch, transformers, fair-esm, biopython, scipy, …) and registers the `mamp-ml` console script.

> If you see `WARNING: There was an error checking the latest version of pip`, ignore it — that's pip's own self-update check, not a problem with the install.

In [ ]:
!pip install --quiet "git+https://github.com/DanielleMStevens/mamp-ml.git@version2"
# Verify the install + see all subcommands.
!mamp-ml --help

## 2. Install a folding backend — do this **now**, before prediction

MAMP-ml needs 3D structures of the receptors, so install a folding backend up front. **Pick ONE** option below and run its install cell. `mamp-ml predict` (step 4) will then fold automatically using whatever you installed — you do **not** run the folding tool by hand.

| Backend | Accuracy | Setup | Speed (T4 GPU) |
|---|---|---|---|
| **A — ColabFold** | production-quality (AlphaFold2) | one-time ~3.6 GB params download | ~5–15 min |
| **B — ESMFold** | slightly lower, usually fine | in-process, no extra env | ~5–15 min |

> Not sure? Start with **ESMFold (Option B)** — it's the fewest moving parts.

### Option A — ColabFold (production-quality, requires a CUDA GPU)

Installs ColabFold and pins JAX 0.4.35. Colab pre-installs a newer `jax_cuda12_plugin`, but `colabfold[alphafold]` downgrades `jaxlib` to 0.5.3, which leaves the plugin incompatible — the runtime then silently falls back to CPU and folding takes hours. The cell below pins the same known-good versions as `scripts/install_colabbatch_linux.sh`.

After this finishes, `colabfold_batch` is on `PATH`; `mamp-ml predict` will auto-detect and run it for you.

In [ ]:
# Install ColabFold + a known-good JAX combination (see the note above).
!pip install --quiet "colabfold[alphafold-minus-jax] @ git+https://github.com/sokrypton/ColabFold"
!pip install --quiet "colabfold[alphafold]"
!pip uninstall -y -q jax jaxlib jax-cuda12-plugin jax-cuda12-pjrt jax-cuda12 jax-plugins
!pip install --quiet --upgrade "jax[cuda12]==0.4.35"
# Download the AF2 monomer-ptm params (~3.6 GB, one-time).
!python -m colabfold.download AlphaFold2-ptm
# Sanity check: jax must see the GPU through the matching CUDA plugin.
!python -c "import jax; print('jax', jax.__version__, '| devices:', jax.devices())"
# Confirm mamp-ml can find the install it will auto-run later.
!mamp-ml find-colabfold

### Option B — ESMFold (in-process, simplest)

ESMFold runs as a regular Python package — no separate env, no JAX pinning. Installing the `esmfold` extra is all you need; `mamp-ml predict --structure esmfold` does the folding in-process. Sequences > 1024 AAs are truncated to the first 1024 (the LRR ectodomain is N-terminal). ~7 GB one-time model download on first fold.

In [ ]:
!pip install --quiet "mamp-ml[esmfold] @ git+https://github.com/DanielleMStevens/mamp-ml.git@version2"

## 3. Upload your input spreadsheet

Columns required: `plant_species`, `receptor`, `locus_id`, `receptor_sequence`, `ligand_sequence`.

> **No data yet?** Skip this cell and smoke-test the whole pipeline on the bundled sample instead: in step 4 run `!mamp-ml predict --example --device cuda` (it ships inside the package). To grab a copy of the sample to edit, run `!mamp-ml example`.

In [ ]:
from google.colab import files
uploaded = files.upload()
input_filename = list(uploaded.keys())[0]
print(f'Using input: {input_filename}')

## 4. Run prediction — one command folds *and* predicts

`mamp-ml predict` runs the entire pipeline end-to-end:

1. builds the receptor FASTA,
2. **folds the receptors automatically** — ColabFold is auto-detected and run for you; ESMFold runs in-process,
3. LRR annotation → B-factor analysis → feature assembly → ESM-2 inference,

writing `predictions.csv` and `lrr_annotation_plots/` into the working directory (the `intermediate_files/` scratch dir is removed). **Run the one cell that matches the backend you installed in step 2.** Each step prints a progress line with a rough time estimate.

> By default only `predictions.csv` and `lrr_annotation_plots/` are kept. Add `--keep all` to retain every intermediate (handy for re-running prediction on a different ligand spreadsheet without re-folding the same receptors).

### ▶ If you installed **ColabFold** (Option A)

In [ ]:
!mamp-ml predict "$input_filename" --device cuda

### ▶ If you installed **ESMFold** (Option B)

In [ ]:
!mamp-ml predict "$input_filename" --structure esmfold --device cuda

## 5. Inspect and download predictions

`predictions.csv` carries the original receptor/ligand metadata plus one column per predicted immunogenicity class.

In [ ]:
import pandas as pd
df = pd.read_csv('predictions.csv')
df.head(10)

In [ ]:
from google.colab import files
files.download('predictions.csv')

---

## Troubleshooting

* **`mamp-ml predict` prints "ColabFold has not been run yet" / "No existing colabfold_batch found":** the Option A install cell didn't finish (or you're on a non-GPU runtime). Re-run it, then `!mamp-ml find-colabfold` should list the install — once it does, `predict` will auto-run it.
* **`jax_cuda12_plugin ... not compatible with the installed jaxlib`:** re-run the Option A install cell; the `pip uninstall` line must land before the final `pip install "jax[cuda12]==0.4.35"`.
* **Folding falls back to CPU / takes hours:** same JAX-pin issue, or no GPU runtime. Check `jax.devices()` shows a GPU (printed by the Option A cell) and that *Runtime → Change runtime type → GPU* is selected.
* **ESM-2 / ESMFold download stalls:** weights are large (ESM-2 ~2.5 GB, ESMFold ~7 GB) and fetched on first use. Retry — the cache persists across cells in a session.
* **CUDA out of memory during ESMFold:** `mamp-ml` auto-retries at smaller trunk chunk sizes; if it still fails, pass `--chunk-size 32` (or `--device cpu` to fall back, slower).
* **Re-use a fold for a different ligand spreadsheet:** run the first prediction with `--keep all`, then swap `input_filename` and re-run `predict` against the same `intermediate_files/` — it detects the existing structures and skips folding.

## Power-user escape hatches

`mamp-ml` exposes one subcommand per pipeline stage if you want to inspect intermediates:

* `mamp-ml prepare <xlsx>` — run the data pipeline only (no inference)
* `mamp-ml example` / `mamp-ml example --path` — copy / locate the bundled sample spreadsheet
* `mamp-ml find-colabfold` — list every `colabfold_batch` install on the machine
* `mamp-ml fold <fasta> <out_dir> --structure esmfold` — fold only
* `mamp-ml prepare-fasta` / `structure-stage` / `lrr-domain-fasta` / `bfactor` / `assemble-test-data` / `chemical-features` — individual stages